In [1]:
import os
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import numpy as np
import pandas as pd

import gzip
import re

ROOT = Path(os.path.abspath(".")).parent

In [ ]:
exp_name = "250716_atac_finetune"
chk = "best_valid_loss"
BASE = f"{ROOT}/Res/{exp_name}/analysis_{chk}/raw_data/var_eff/"

label_meta = pd.read_csv(f"{ROOT}/Data/lc_HM_ATAC_v1/label_meta.csv", index_col=1)
vcf = pd.read_csv(f"{ROOT}/Data/source/gwas.catlog.vcf", sep="\t")

In [ ]:
vcf

# Functions

## BigWig Writing Functions

In [ ]:
import pyBigWig
from itertools import groupby

def get_chrom_sizes(fasta_file):
    """
    Get chromosome sizes from FASTA file.

    Parameters:
    -----------
    fasta_file : str
        Path to reference genome FASTA file

    Returns:
    --------
    dict : Dictionary mapping chromosome names to sizes
    """
    chrom_sizes = {}
    current_chrom = None
    current_size = 0

    with open(fasta_file, 'r') as f:
        for line in f:
            if line.startswith('>'):
                # Save previous chromosome
                if current_chrom is not None:
                    chrom_sizes[current_chrom] = current_size
                # Start new chromosome
                current_chrom = line.strip().split()[0][1:]  # Remove '>'
                current_size = 0
            else:
                current_size += len(line.strip())

        # Save last chromosome
        if current_chrom is not None:
            chrom_sizes[current_chrom] = current_size

    return chrom_sizes


def write_predictions_to_bigwig(predictions, chrom, region_start, region_end, 
                                  output_file, chrom_sizes, window_size=32):
    """
    Write predictions for a specific region to a BigWig file.

    Parameters:
    -----------
    predictions : np.ndarray
        Predictions array of shape (n_windows,)
    chrom : str
        Chromosome name
    region_start : int
        Start position of the region
    region_end : int
        End position of the region
    output_file : str
        Output BigWig file path
    chrom_sizes : dict
        Dictionary mapping chromosome names to sizes
    window_size : int
        Size of genomic bins in bp (default: 32)
    """
    n_windows = len(predictions)
    
    # Calculate the actual genomic range covered by predictions
    pred_length = n_windows * window_size
    
    # Center the predictions on the region
    pred_start = region_start + (region_end - region_start - pred_length) // 2
    
    # Collect all entries
    all_entries = []
    for i, value in enumerate(predictions):
        window_start = pred_start + i * window_size
        window_end = window_start + window_size
        
        # Only include windows that fall within reasonable chromosome bounds
        if window_start >= 0 and window_end <= chrom_sizes.get(chrom, 300000000):
            all_entries.append((chrom, int(window_start), int(window_end), float(value)))
    
    # Sort entries by chromosome and start position
    chrom_order = {chrom: i for i, chrom in enumerate(chrom_sizes.keys())}
    all_entries.sort(key=lambda x: (chrom_order.get(x[0], 999), x[1]))
    
    # Create BigWig file
    bw = pyBigWig.open(output_file, "w")
    bw.addHeader(list(chrom_sizes.items()))
    
    # Write entries chromosome by chromosome
    for chrom_name, chrom_entries in groupby(all_entries, key=lambda x: x[0]):
        chrom_entries = list(chrom_entries)
        if len(chrom_entries) > 0:
            chroms = [e[0] for e in chrom_entries]
            starts = [e[1] for e in chrom_entries]
            ends = [e[2] for e in chrom_entries]
            values = [e[3] for e in chrom_entries]
            bw.addEntries(chroms, starts, ends=ends, values=values)
    
    bw.close()
    print(f"Created: {output_file}")

In [ ]:
def search_anno(target_gene, target_transcript=None):

    gtf_path = f"{ROOT}/Data/source/gencode.v48.annotation.gtf.gz"

    exons = []
    with gzip.open(gtf_path, "rt") as fp:
        for line in fp:
            if line.startswith("#"):
                continue
            cols = line.rstrip("\n").split("\t")
            if cols[2] != "exon":
                continue
            attr_str = cols[8]
            if f'gene_name "{target_gene}"' not in attr_str:
                continue

            # 解析 attributes
            attrs = dict(re.findall(r'(\S+) "([^"]+)"', attr_str))
            tx_id = attrs.get("transcript_id")
            if target_transcript and tx_id != target_transcript:
                continue

            start, end = int(cols[3]), int(cols[4])
            exons.append((start, end))

    exons.sort(key=lambda x: x[0])

    annos = {}
    for i, (start, end) in enumerate(exons, start=1):
        key = target_gene if i == 1 else f"{target_gene}.{i}"
        annos[key] = {"idx": (start, end), "show_text": False}

    return annos

In [ ]:
def prepare_data(track, var_idx, annos):

    track_dim = label_meta.loc[track, "dim"]
    chr_name, pos, ref, alt = vcf.iloc[var_idx, [0, 1, 3, 4]]

    annos.update({"Mut": {"idx": (pos, pos), "show_text": True}})

    with h5py.File(f"{BASE}/{chr_name}_{ref}{pos}{alt}.h5", "r") as f:

        data = {
            "label": f["data/label"][:, track_dim],
            "pred_wt": f["data/pred_wt"][:, track_dim],
            "pred_alt": f["data/pred_alt"][:, track_dim],
            "diff": f["data/diff"][:, track_dim],
        }

        metadata = {
            "context_start": f.attrs["context_start"],
            "context_end": f.attrs["context_end"],
        }

    # add basic plot track data
    plot_data = list(data.values())
    plot_title = ["Target", "Wt Pred", "Mut Pred", "Diff (Raw)"]

    trim = (524288 // 32 - data["diff"].shape[0]) // 2
    x = np.arange(metadata["context_start"], metadata["context_end"]).reshape(-1, 32)[trim:-trim]
    for k, v in annos.items():
        start, end = v["idx"]
        bin_start = np.where(x == start)[0]
        bin_end = np.where(x == end)[0]
        v["bin"] = (bin_start, bin_end)

    return data, plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track

In [ ]:
def plot(plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track):

    # start to plot
    n = len(plot_data) + 1
    height_ratios = [1] * len(plot_data) + [0.7]
    fig, axes = plt.subplots(
        nrows=n, ncols=1, figsize=(8, n * 1.5), sharex=True, gridspec_kw={"height_ratios": height_ratios}
    )

    x = x[:, 0]

    # add tracks
    for i, ax in enumerate(axes[:-1]):
        ax.plot(x, plot_data[i])
        ax.set_title(plot_title[i])
        ax.set_ylabel(None)
        ax.set_xlabel(None)

    # add annos
    ax = axes[-1]
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_ylabel(None)
    ax.set_xlabel(None)

    linewidth = 1.5
    rec_width = 0.2
    ax.hlines(0.5, x.min(), x.max(), color="black", linewidth=linewidth)
    ax.set_title(f"Chromosome {chr_name[3:]}, {ref} {pos} {alt}, {track}")

    for i, (k, v) in enumerate(annos.items()):
        # start‐ and end‐positions as scalars
        x0 = x[v["bin"][0]].item()
        width = (x[v["bin"][1]] - x[v["bin"][0]]).item()

        # draw the rectangle
        rect = Rectangle(
            (x0, 0.5 - 0.5 * rec_width),
            width,
            rec_width,
            facecolor="lightblue",
            edgecolor="black",
            linewidth=linewidth,
        )
        ax.add_patch(rect)

        # add the label 'k' centered on top of the rectangle
        if v["show_text"]:
            ax.text(
                x0 + width / 2,  # x‐position: middle of the rect
                0.5 + rec_width / 2 + 0.04,  # y‐position: just above the rect
                k,  # the annotation text
                ha="center",  # horizontal alignment
                va="bottom",  # vertical alignment
                fontsize="large",
                rotation=0,
            )

    plt.tight_layout()
    plt.show()

# Case

In [ ]:
annos = {}

In [ ]:
target_gene = "CACNA1C"
target_transcript = "ENST00000682544.1"
annos.update(search_anno(target_gene, target_transcript))

In [ ]:
annos["CACNA1C"]["show_text"] = True

In [ ]:
annos

Case

In [ ]:
track = "STR-D1-MSN-GABA_K27Ac"

data, plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track = prepare_data(
    track=track, var_idx=7, annos=annos
)

to_remove = []
for bin in annos.keys():
    if bin.startswith("CACNA1C"):
        if len(annos[bin]['bin'][0]) == 0:
            print(bin)
            to_remove.append(bin)
for bin in to_remove:
    annos.pop(bin)
    
plot(plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track)

Control

In [ ]:
track = "Microglia_K27Ac"

data, plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track = prepare_data(
    track=track, var_idx=4, annos=annos
)

plot(plot_data, plot_title, x, annos, chr_name, ref, pos, alt, track)

In [ ]:
# Write BigWig files for the current variant and track
# This will create separate BigWig files for label, pred_wt, pred_alt, and diff

track_types = ["label", "pred_wt", "pred_alt", "diff"]
window_size = 32

for track_type in track_types:
    predictions = data[track_type]
    
    # Create output filename
    safe_track = track.replace('/', '_').replace(' ', '_')
    output_file = f"{output_dir}/{chr_name}_{ref}{pos}{alt}_{safe_track}_{track_type}.bw"
    
    # Write BigWig file
    write_predictions_to_bigwig(
        predictions=predictions,
        chrom=chr_name,
        region_start=metadata['context_start'],
        region_end=metadata['context_end'],
        output_file=output_file,
        chrom_sizes=chrom_sizes,
        window_size=window_size
    )
    
print(f"\nAll BigWig files written to: {output_dir}")

In [ ]:
# Setup for BigWig writing
fasta_file = f"{ROOT}/Data/Ref/hg38/hg38.fa"
chrom_sizes = get_chrom_sizes(fasta_file)
print(f"Loaded chromosome sizes for {len(chrom_sizes)} chromosomes")

# Create output directory
output_dir = f"{BASE}/bigwig"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

# Write BigWig Files